# SEC EDGAR Financial Data Pipeline
Pulls quarterly R&D, Revenue, and Cost of Revenue from the SEC EDGAR XBRL API
for companies in our cleaned patent database.

**Outputs written to this directory:**
- `sec_filers.csv` – companies matched to a SEC CIK
- `non_sec_filers.csv` – companies with no SEC filing found
- `sec_financials.csv` – quarterly financial data matching the screenshot format

In [ ]:
import requests, json, time, re
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# SEC requires a descriptive User-Agent
HEADERS = {
    'User-Agent': 'Energy Economics Lab research@energyeconlab.edu',
    'Accept-Encoding': 'gzip, deflate'
}
SLEEP = 0.22   # ~4.5 req/s — below SEC's 10/s hard limit
BASE  = Path('/Users/humza/energy_economics_lab_work/Energy_Econ_Lab/Company R&D')
CLEANED_CSV = '/Users/humza/Downloads/companies_cleaned.csv'

In [ ]:
# Hardcoded CIKs (verified against EDGAR).
# Format: 'Official Name': (cik_str, primary_form_type)
# None = known non-filer; skips EDGAR lookup.
KNOWN_CIKS = {
    # US domestic 10-K / 10-Q filers
    'Nvidia Corporation':                        ('1045810',  '10-K/10-Q'),
    'Intel Corporation':                         ('50863',    '10-K/10-Q'),
    'Micron Technology, Inc.':                   ('723125',   '10-K/10-Q'),
    'Advanced Micro Devices, Inc.':              ('2488',     '10-K/10-Q'),
    'Applied Materials, Inc.':                   ('6951',     '10-K/10-Q'),
    'Qualcomm Incorporated':                     ('804328',   '10-K/10-Q'),
    'Texas Instruments Incorporated':            ('97476',    '10-K/10-Q'),
    'Alphabet Inc.':                             ('1652044',  '10-K/10-Q'),
    'Microsoft Corporation':                     ('789019',   '10-K/10-Q'),
    'Apple Inc.':                                ('320193',   '10-K/10-Q'),
    'Amazon.com, Inc.':                          ('1018724',  '10-K/10-Q'),
    'Meta Platforms, Inc.':                      ('1326801',  '10-K/10-Q'),
    'Dell Technologies Inc.':                    ('1571996',  '10-K/10-Q'),
    'IBM Corporation':                           ('51143',    '10-K/10-Q'),
    'Western Digital Corporation':               ('106040',   '10-K/10-Q'),
    'Lam Research Corporation':                  ('707549',   '10-K/10-Q'),
    'Cisco Systems, Inc.':                       ('858877',   '10-K/10-Q'),
    'Oracle Corporation':                        ('1341439',  '10-K/10-Q'),
    'Adobe Inc.':                                ('796343',   '10-K/10-Q'),
    'Salesforce, Inc.':                          ('1108524',  '10-K/10-Q'),
    'VMware LLC':                                ('1124610',  '10-K/10-Q'),
    'Snap Inc.':                                 ('1564408',  '10-K/10-Q'),
    'PayPal Holdings, Inc.':                     ('1633917',  '10-K/10-Q'),
    'Visa Inc.':                                 ('1403161',  '10-K/10-Q'),
    'Mastercard Incorporated':                   ('1141391',  '10-K/10-Q'),
    'Wells Fargo & Company':                     ('72971',    '10-K/10-Q'),
    'JPMorgan Chase & Co.':                      ('19617',    '10-K/10-Q'),
    'Bank of America Corporation':               ('70858',    '10-K/10-Q'),
    'Capital One Financial Corporation':         ('927628',   '10-K/10-Q'),
    'Honeywell International Inc.':              ('773840',   '10-K/10-Q'),
    'GE Aerospace':                              ('40987',    '10-K/10-Q'),
    'Carrier Global Corporation':                ('1783398',  '10-K/10-Q'),
    'GlobalFoundries Inc.':                      ('1709048',  '10-K/10-Q'),
    'Wolfspeed, Inc.':                           ('895419',   '10-K/10-Q'),
    'Monolithic Power Systems, Inc.':            ('1280452',  '10-K/10-Q'),
    'onsemi (ON Semiconductor)':                 ('1097864',  '10-K/10-Q'),
    'Microchip Technology Incorporated':         ('827054',   '10-K/10-Q'),
    'Synopsys, Inc.':                            ('883241',   '10-K/10-Q'),
    'Cadence Design Systems, Inc.':              ('813672',   '10-K/10-Q'),
    'Silicon Laboratories Inc.':                 ('1038074',  '10-K/10-Q'),
    'MACOM Technology Solutions Holdings, Inc.': ('1493594',  '10-K/10-Q'),
    'ServiceNow, Inc.':                          ('1373715',  '10-K/10-Q'),
    'Pure Storage, Inc.':                        ('1474735',  '10-K/10-Q'),
    'Snowflake Inc.':                            ('1640147',  '10-K/10-Q'),
    'NetApp, Inc.':                              ('1002047',  '10-K/10-Q'),
    'Arista Networks, Inc.':                     ('1596532',  '10-K/10-Q'),
    'Juniper Networks, Inc.':                    ('1043604',  '10-K/10-Q'),
    'Ciena Corporation':                         ('936395',   '10-K/10-Q'),
    'Corning Incorporated':                      ('24741',    '10-K/10-Q'),
    'Equifax Inc.':                              ('33185',    '10-K/10-Q'),
    'Fair Isaac Corporation (FICO)':             ('814547',   '10-K/10-Q'),
    'Palo Alto Networks, Inc.':                  ('1327567',  '10-K/10-Q'),
    'Illumina, Inc.':                            ('1110803',  '10-K/10-Q'),
    'eBay Inc.':                                 ('1065088',  '10-K/10-Q'),
    'Dropbox, Inc.':                             ('1467623',  '10-K/10-Q'),
    'Intuit Inc.':                               ('896878',   '10-K/10-Q'),
    'Tesla, Inc.':                               ('1318605',  '10-K/10-Q'),
    'Seagate Technology Holdings plc':           ('1137789',  '10-K/10-Q'),
    'Kyndryl Holdings, Inc.':                    ('1867072',  '10-K/10-Q'),
    'Allegro MicroSystems, Inc.':                ('866291',   '10-K/10-Q'),
    'Power Integrations, Inc.':                  ('833640',   '10-K/10-Q'),
    'Navitas Semiconductor Limited':             ('1801170',  '10-K/10-Q'),
    'Analog Devices, Inc.':                      ('6951',     '10-K/10-Q'),
    'Skyworks Solutions, Inc.':                  ('4127',     '10-K/10-Q'),
    'Booz Allen Hamilton Inc.':                  ('1443646',  '10-K/10-Q'),
    'Palantir Technologies Inc.':                ('1321655',  '10-K/10-Q'),
    'Rambus Inc.':                               ('917273',   '10-K/10-Q'),
    'Marvell Technology, Inc.':                  ('1058057',  '10-K/10-Q'),
    'Entegris, Inc.':                            ('1101302',  '10-K/10-Q'),
    'Diodes Incorporated':                       ('29002',    '10-K/10-Q'),
    'Adeia Inc.':                                ('1840292',  '10-K/10-Q'),
    'Accenture plc':                             ('1467373',  '10-K/10-Q'),
    'Broadcom Inc.':                             ('1730168',  '10-K/10-Q'),
    'NXP Semiconductors N.V.':                   ('1413447',  '10-K/10-Q'),
    'Alpha and Omega Semiconductor Limited':     ('1417398',  '10-K/10-Q'),
    'D-Wave Systems Inc.':                       ('1907982',  '10-K/10-Q'),
    'Veeco Instruments Inc.':                    ('103145',   '10-K/10-Q'),
    # Pre-revenue / R&D-stage companies — correct CIKs; filtered out post-fetch
    'atomera incorporated':                      ('1420520',  '10-K/10-Q'),
    'aurora innovation inc':                     ('1828108',  '10-K/10-Q'),
    'hyliion holdings corp':                     ('1759631',  '10-K/10-Q'),
    'ideal power inc':                           ('1507957',  '10-K/10-Q'),
    'lightwave logic inc':                       ('1325964',  '10-K/10-Q'),
    'quantum computing inc':                     ('1758009',  '10-K/10-Q'),
    # Foreign filers — 20-F annual (IFRS; our code checks ifrs-full namespace)
    'Taiwan Semiconductor Manufacturing Company Limited': ('1046179', '20-F'),
    'Arm Holdings plc':                          ('1973239',  '20-F'),
    # Foreign filers that file domestic 10-K/10-Q (not 20-F)
    'STMicroelectronics N.V.':                   ('932787',   '10-K/10-Q'),
    'Ericsson':                                  ('717826',   '10-K/10-Q'),
    # No EDGAR XBRL — handled by Yahoo Finance fallback cell below
    'SAP SE':                                    None,
    'Sony Group Corporation':                    None,
    'ASE Technology Holding Co., Ltd.':          None,
    'United Microelectronics Corporation':       None,
    'tower semiconductor ltd':                   None,
    'Canon Inc.':                                None,
    # Known non-filers
    'Samsung Electronics Co., Ltd.':            None,
    'SK Hynix Inc.':                            None,
    'Huawei Technologies Co., Ltd.':            None,
    'Kioxia Corporation':                        None,
    'Nanya Technology Corporation':             None,
    'Mitsubishi Electric Corporation':          None,
    'DENSO Corporation':                         None,
    'Murata Manufacturing Co., Ltd.':           None,
    'Tokyo Electron Limited':                    None,
    'Renesas Electronics Corporation':           None,
    'Hitachi, Ltd.':                            None,
    'Fujitsu Limited':                           None,
    'Toshiba Corporation':                       None,
    'Infineon Technologies AG':                  None,
    'Robert Bosch GmbH':                        None,
    'MediaTek Inc.':                             None,
    'Tencent Holdings Limited':                  None,
    'Baidu, Inc.':                               None,
    'Yangtze Memory Technologies Co., Ltd.':    None,
    'Semiconductor Manufacturing International Corporation': None,
    'Changxin Memory Technologies, Inc.':        None,
}

In [ ]:
df_co = pd.read_csv(CLEANED_CSV)
print(f'{len(df_co):,} canonical companies loaded')
df_co.head(10)

In [ ]:
# Download EDGAR company ticker list — one call, ~1 MB, covers all SEC registrants.
print('Fetching EDGAR tickers...')
r = requests.get('https://www.sec.gov/files/company_tickers.json', headers=HEADERS)
r.raise_for_status()

def _norm(s):
    s = s.lower()
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

edgar_lookup = {}   # normalised_name -> (cik_str, ticker, title)
for entry in r.json().values():
    norm = _norm(entry['title'])
    edgar_lookup[norm] = (str(entry['cik_str']), entry.get('ticker', ''), entry['title'])

print(f'{len(edgar_lookup):,} EDGAR companies indexed')

In [ ]:
def find_cik(name):
    # 1. Hardcoded dict
    if name in KNOWN_CIKS:
        val = KNOWN_CIKS[name]
        if val is None:
            return None, None, 'known non-filer'
        return val[0], val[1], 'hardcoded'
    # 2. Exact normalised name match
    norm = _norm(name)
    if norm in edgar_lookup:
        return edgar_lookup[norm][0], '10-K/10-Q', 'edgar-exact'
    # 3. Partial match (handles Inc./LLC suffix variants)
    for key, val in edgar_lookup.items():
        if len(key) > 10 and norm.startswith(key[:25]):
            return val[0], '10-K/10-Q', 'edgar-partial'
    return None, None, 'not found'

results = []
for _, row in df_co.iterrows():
    cik, ftype, source = find_cik(row['official_name'])
    results.append({
        'official_name':   row['official_name'],
        'entity_type':     row['entity_type'],
        'total_patents':   row['total_patents'],
        'total_citations': row['total_citations'],
        'notes':           row['notes'],
        'raw_variants':    row['raw_name_variants'],
        'cik':             cik,
        'form_type':       ftype,
        'cik_source':      source,
    })

df_matched     = pd.DataFrame(results)
sec_filers     = df_matched[df_matched['cik'].notna()].copy()
non_sec_filers = df_matched[df_matched['cik'].isna()].copy()

print(f'SEC filers found : {len(sec_filers):,}')
print(f'No SEC filing    : {len(non_sec_filers):,}')
sec_filers[['official_name', 'cik', 'form_type', 'cik_source', 'total_patents']].head(20)

In [ ]:
sec_filers.to_csv(BASE / 'sec_filers.csv', index=False)
non_sec_filers.to_csv(BASE / 'non_sec_filers.csv', index=False)
print('Saved sec_filers.csv and non_sec_filers.csv')

In [ ]:
# Revenue tag priority order — most specific / most complete first.
# merge_all_tags() unions data across ALL of these, so every era is covered.
REVENUE_TAGS = [
    # Post-ASC 606 (adopted ~2018) — most companies' primary tag today
    'RevenueFromContractWithCustomerExcludingAssessedTax',
    'RevenueFromContractWithCustomerIncludingAssessedTax',
    # Pre-ASC 606 names — cover history before the transition
    'SalesRevenueNet',
    'SalesRevenueGoodsNet',
    'SalesRevenueServicesNet',
    # Broad aggregate used by some companies across all eras
    'Revenues',
    'NetRevenues',
    # Payment-network companies (Visa, Mastercard report net-of-interchange)
    'RevenuesNetOfInterestExpense',
    # Banks and financial institutions
    'InterestAndDividendIncomeOperating',
    'NoninterestIncome',
]
COR_TAGS = [
    'CostOfRevenue',
    'CostOfGoodsAndServicesSold',
    'CostOfGoodsSold',
    'CostOfServices',
]
RND_TAGS = [
    'ResearchAndDevelopmentExpense',
    'ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost',
]
# IFRS equivalents — used by most foreign 20-F filers (TSMC, STMicro, SAP, etc.)
IFRS_REVENUE_TAGS = [
    'Revenue',
    'RevenueFromContractsWithCustomers',
]
IFRS_COR_TAGS = [
    'CostOfSales',
]
IFRS_RND_TAGS = [
    'ResearchAndDevelopmentExpense',
]

def _pad_cik(cik): return str(cik).zfill(10)

def _filing_url(cik, accn):
    return f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accn.replace('-', '')}/"

def _best_period_fact(facts, td, tol=30):
    """
    Pick the fact with the LATEST end date among those whose period
    length is within tol days of td.  This selects the current reporting
    period, not the prior-year comparison included in the same filing.
    """
    candidates = []
    for f in facts:
        if 'start' not in f:
            continue
        try:
            days = (pd.to_datetime(f['end']) - pd.to_datetime(f['start'])).days
        except Exception:
            continue
        if abs(days - td) <= tol:
            candidates.append(f)
    if not candidates:
        return None
    return max(candidates, key=lambda f: f['end'])

def extract_metric(usd_facts, forms, target_days):
    """Return {accn: {val, filed, end, form}} — one entry per filing."""
    by_accn = {}
    for f in usd_facts:
        if f.get('form') not in forms:
            continue
        by_accn.setdefault(f['accn'], []).append(f)
    out = {}
    for accn, facts in by_accn.items():
        chosen = _best_period_fact(facts, target_days)
        if chosen:
            out[accn] = {'val': chosen['val'], 'filed': chosen.get('filed', ''),
                         'end': chosen['end'], 'form': chosen['form']}
    return out

def first_available(gaap, tags, forms, target_days):
    """Try each XBRL tag in order; return the first non-empty result."""
    for tag in tags:
        facts = gaap.get(tag, {}).get('units', {}).get('USD', [])
        if facts:
            result = extract_metric(facts, forms, target_days)
            if result:
                return result, tag
    return {}, None

def merge_all_tags(gaap, tags, forms, target_days):
    """
    Union facts across ALL tags into one {accn: record} dict.
    When the same accn appears under multiple tags, the first (highest-
    priority) tag wins.  This covers the ASC 606 transition (~2018) where
    companies switched from SalesRevenueNet / Revenues to
    RevenueFromContractWithCustomerExcludingAssessedTax mid-history,
    leaving gaps if only the first tag with any data is used.
    """
    merged = {}
    for tag in tags:
        facts = gaap.get(tag, {}).get('units', {}).get('USD', [])
        if not facts:
            continue
        result = extract_metric(facts, forms, target_days)
        for accn, rec in result.items():
            if accn not in merged:   # first (highest-priority) tag wins
                merged[accn] = rec
    return merged

def fetch_company_financials(cik, company_name, forms):
    """Fetch XBRL facts for one company. Returns list of row dicts."""
    target_days = 365 if forms == {'20-F'} else 91
    url = f'https://data.sec.gov/api/xbrl/companyfacts/CIK{_pad_cik(cik)}.json'
    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        time.sleep(SLEEP)
        resp.raise_for_status()
    except Exception as e:
        print(f'  ERROR {company_name}: {e}')
        return []
    facts = resp.json().get('facts', {})
    gaap  = facts.get('us-gaap', {})
    ifrs  = facts.get('ifrs-full', {})
    if gaap:
        rnd_data, _ = first_available(gaap, RND_TAGS,      forms, target_days)
        rev_data     = merge_all_tags(gaap, REVENUE_TAGS,  forms, target_days)
        cor_data, _ = first_available(gaap, COR_TAGS,      forms, target_days)
    elif ifrs:
        # Foreign 20-F filers reporting under IFRS (e.g. TSMC, STMicro, SAP)
        rnd_data, _ = first_available(ifrs, IFRS_RND_TAGS,     forms, target_days)
        rev_data     = merge_all_tags(ifrs, IFRS_REVENUE_TAGS, forms, target_days)
        cor_data, _ = first_available(ifrs, IFRS_COR_TAGS,     forms, target_days)
    else:
        return []
    rows = []
    for accn in set(rnd_data) | set(rev_data) | set(cor_data):
        meta = rnd_data.get(accn) or rev_data.get(accn) or cor_data.get(accn)
        rows.append({
            'Company':         company_name,
            'Form Type':       meta['form'],
            'Filing date':     meta['filed'],
            'Period end':      meta['end'],
            'R&D Expenses':    round(rnd_data[accn]['val'] / 1e6, 2) if accn in rnd_data else None,
            'Revenue':         round(rev_data[accn]['val'] / 1e6, 2) if accn in rev_data else None,
            'Cost of Revenue': round(cor_data[accn]['val'] / 1e6, 2) if accn in cor_data else None,
            'URL':             _filing_url(cik, accn),
        })
    return rows

print('Helpers defined.')

In [ ]:
# Verify against the screenshot: Nvidia 2017-05-23 should show R&D=411, Rev=1937, CoR=787
test = fetch_company_financials('1045810', 'Nvidia Corporation', {'10-Q'})
df_test = pd.DataFrame(test).sort_values('Filing date')
print(f'Nvidia: {len(df_test)} quarterly filings')
df_test[['Company', 'Form Type', 'Filing date', 'R&D Expenses', 'Revenue', 'Cost of Revenue']].head(10)

In [ ]:
# Fetch financials for all SEC filers.
# Results are cached in edgar_cache.json so subsequent runs skip companies
# already fetched.  Set FORCE_REFRESH = True to re-fetch everything.
CACHE_FILE    = BASE / 'edgar_cache.json'
FORCE_REFRESH = False

if CACHE_FILE.exists() and not FORCE_REFRESH:
    with open(CACHE_FILE) as _f:
        edgar_cache = json.load(_f)
    print(f'Cache loaded — {len(edgar_cache)} companies already fetched')
else:
    edgar_cache = {}
    if FORCE_REFRESH:
        print('FORCE_REFRESH=True — ignoring existing cache')

all_rows, failed = [], []

for _, row in sec_filers.iterrows():
    name  = row['official_name']
    cik   = str(row['cik'])
    ftype = str(row['form_type'])
    forms = {'20-F'} if '20-F' in ftype and '10' not in ftype else {'10-Q'}

    if cik in edgar_cache:
        cached = edgar_cache[cik]      # None = previously fetched, no data
        if cached:
            all_rows.extend(cached)
            print(f'  {name} ... {len(cached)} periods  [cache]')
        else:
            failed.append(name)
            print(f'  {name} ... no data  [cache]')
        continue

    print(f'  {name} (CIK {cik}) ...', end=' ')
    rows = fetch_company_financials(cik, name, forms)
    if rows:
        all_rows.extend(rows)
        edgar_cache[cik] = rows
        print(f'{len(rows)} periods')
    else:
        failed.append(name)
        edgar_cache[cik] = None   # cache the miss so we don't hit EDGAR again
        print('no data')

# Persist cache for next run
with open(CACHE_FILE, 'w') as _f:
    json.dump(edgar_cache, _f)

print(f'\nTotal rows: {len(all_rows):,}  |  Cache saved ({len(edgar_cache)} companies)')
if failed:
    print('No data found for:', failed)

In [ ]:
import yfinance as yf

# Foreign companies not available via SEC EDGAR XBRL.
# Financial data is fetched from Financial Modeling Prep (FMP), which
# provides up to 40 quarters of history.  Non-USD currencies (JPY, TWD, …)
# are converted to USD using the historical exchange rate on each specific
# reporting date.  Paste your FMP API key into FMP_API_KEY below.
# Add more entries here as needed: 'Canonical Name': ('TICKER', 'Country')
FOREIGN_YFINANCE = {
    'Tower Semiconductor Ltd.':             ('TSEM', 'Israel'),
    'ASML Holding N.V.':                    ('ASML', 'Netherlands'),
    'SAP SE':                               ('SAP',  'Germany'),
    'Sony Group Corporation':               ('SONY', 'Japan'),
    'United Microelectronics Corporation':  ('UMC',  'Taiwan'),
    'ASE Technology Holding Co., Ltd.':     ('ASX',  'Taiwan'),
}

# Companies we mark as foreign for the 'Foreign' column (EDGAR + yfinance)
FOREIGN_NAMES = (
    {name for name, val in KNOWN_CIKS.items()
     if isinstance(val, tuple) and val[1] == '20-F'}
    | {name for name, val in KNOWN_CIKS.items()
       if isinstance(val, tuple) and val[1] == '10-K/10-Q'
       and name in ('STMicroelectronics N.V.', 'Ericsson')}
    | set(FOREIGN_YFINANCE.keys())
)

# ── FMP API key ───────────────────────────────────────────────────────────────
FMP_API_KEY = '4Vj0A49V4a2cg92ymoHEMQzwLv0AVZIu'

def _make_fx_converter(fin_currency):
    """
    Return a callable  date_str -> float  giving the USD conversion rate
    on (or just before) the supplied date.

    * USD / EUR : Yahoo Finance already reports these in USD, so the
      converter always returns 1.0.
    * Other currencies (JPY, TWD, …): fetch the full available history
      from Yahoo Finance ({CUR}USD=X) and use pandas .asof() so each
      reporting period gets the exchange rate from *that specific date*
      rather than a single average.

    Returns None if the FX history cannot be fetched.
    """
    if fin_currency in ('USD', 'EUR'):   # Yahoo auto-converts EUR→USD
        return lambda _: 1.0
    try:
        hist = yf.Ticker(f'{fin_currency}USD=X').history(period='max')
        if hist.empty:
            return None
        series = hist['Close'].copy()
        # Strip timezone so Timestamp comparisons work without tz mismatch
        series.index = pd.to_datetime(series.index).tz_localize(None)
        series = series.sort_index()
        def _fx(date_str):
            dt   = pd.Timestamp(date_str)
            rate = series.asof(dt)
            if pd.isna(rate):            # date predates history — use earliest known
                rate = series.iloc[0]
            return float(rate)
        return _fx
    except Exception:
        return None

def fetch_fmp_financials(company_name, ticker):
    """
    Fetch up to 40 quarters of income-statement data from Financial Modeling
    Prep (FMP).  The `reportedCurrency` field in each period is used to
    convert non-USD values to USD using the historical FX rate on that date.
    Requires FMP_API_KEY to be set above.
    """
    endpoint = (
        f'https://financialmodelingprep.com/api/v3/income-statement/{ticker}'
        f'?period=quarter&limit=40&apikey={FMP_API_KEY}'
    )
    try:
        resp = requests.get(endpoint, timeout=30)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        print(f'  FMP error for {company_name}: {e}')
        return []
    # FMP returns a list on success; a dict (with 'Error Message') on failure
    if not data or isinstance(data, dict):
        print(f'  FMP: no data returned for {company_name} ({ticker})')
        return []
    # Cache FX converters per currency so we only fetch history once
    fx_cache = {}
    rows = []
    for period in data:
        date_str = str(period.get('date', ''))[:10]
        if not date_str:
            continue
        currency = period.get('reportedCurrency', 'USD')
        if currency not in fx_cache:
            fx_cache[currency] = _make_fx_converter(currency)
        fx_converter = fx_cache[currency]
        if fx_converter is None:
            continue   # cannot get FX rate — skip this period
        fx = fx_converter(date_str)
        def _v(key):
            val = period.get(key)
            if val is None:
                return None
            return round(float(val) * fx / 1e6, 2)
        rev = _v('revenue')
        rnd = _v('researchAndDevelopmentExpenses')
        cor = _v('costOfRevenue')
        if rev is None and rnd is None and cor is None:
            continue
        rows.append({
            'Company':         company_name,
            'Form Type':       'Quarterly',
            'Filing date':     date_str,
            'Period end':      date_str,
            'R&D Expenses':    rnd,
            'Revenue':         rev,
            'Cost of Revenue': cor,
            'URL':             f'https://financialmodelingprep.com/financial-statements/{ticker}',
            'Data Source':     'FMP',
        })
    return rows

def fetch_yfinance_financials(company_name, ticker):
    """
    Fetch BOTH annual and quarterly income-statement data from Yahoo Finance
    and merge them to maximise historical coverage:
      - quarterly_income_stmt  →  last ~4–5 quarters (granular, recent)
      - income_stmt            →  last ~4 annual periods (older history)
    Annual periods are included only for fiscal years not already covered
    by quarterly data, so there is no double-counting.
    For non-USD/EUR currencies each period uses the exchange rate on that
    specific reporting date (via _make_fx_converter).
    """
    co = yf.Ticker(ticker)
    fin_currency = co.info.get('financialCurrency', 'USD')
    fx_converter = _make_fx_converter(fin_currency)
    if fx_converter is None:
        print(f'  {company_name}: cannot get FX rate for {fin_currency}')
        return []
    url = f'https://finance.yahoo.com/quote/{ticker}/financials'

    def _stmt_rows(stmt, freq):
        """Convert one income-statement DataFrame into a list of row dicts."""
        out = []
        if stmt is None or stmt.empty:
            return out
        for col in stmt.columns:
            date_str = col.strftime('%Y-%m-%d') if hasattr(col, 'strftime') else str(col)[:10]
            fx = fx_converter(date_str)
            def _v(names):
                for n in names:
                    if n in stmt.index and pd.notna(stmt.loc[n, col]):
                        return round(float(stmt.loc[n, col]) * fx / 1e6, 2)
                return None
            rev = _v(['Total Revenue', 'Operating Revenue'])
            rnd = _v(['Research And Development'])
            cor = _v(['Cost Of Revenue'])
            if rev is None and rnd is None and cor is None:
                continue
            out.append({
                'Company':         company_name,
                'Form Type':       freq,
                'Filing date':     date_str,
                'Period end':      date_str,
                'R&D Expenses':    rnd,
                'Revenue':         rev,
                'Cost of Revenue': cor,
                'URL':             url,
                'Data Source':     'Yahoo Finance',
            })
        return out

    quarterly_rows = _stmt_rows(co.quarterly_income_stmt, 'Quarterly')
    annual_rows    = _stmt_rows(co.income_stmt,           'Annual')

    # Keep annual periods only for fiscal years not already covered quarterly
    q_years      = {r['Period end'][:4] for r in quarterly_rows}
    older_annual = [r for r in annual_rows if r['Period end'][:4] not in q_years]
    return older_annual + quarterly_rows

foreign_rows = []
for co_name, (ticker, country) in FOREIGN_YFINANCE.items():
    print(f'  {co_name} ({ticker}) ...', end=' ')
    rows = fetch_fmp_financials(co_name, ticker)
    foreign_rows.extend(rows)
    print(f'{len(rows)} periods')

print(f'\nFMP rows: {len(foreign_rows)}')

In [ ]:
# For every company that got a 404 or returned no data from EDGAR,
# look up its ticker from the EDGAR tickers index already in memory
# and attempt a Yahoo Finance fetch.  Companies already handled by
# cell-yfinance (FOREIGN_YFINANCE) are skipped.

# Reverse lookup: CIK string -> ticker symbol
cik_to_ticker = {v[0]: v[1] for v in edgar_lookup.values() if v[1]}

already_yfin  = set(FOREIGN_YFINANCE.keys())
fallback_rows = []
still_no_data = []

if not failed:
    print('No failed companies — nothing to retry.')
else:
    failed_info = (
        sec_filers[sec_filers['official_name'].isin(set(failed))]
        [['official_name', 'cik']].drop_duplicates()
    )
    print(f'Retrying {len(failed_info)} companies via Yahoo Finance...\n')

    for _, row in failed_info.iterrows():
        name   = row['official_name']
        cik    = str(row['cik'])

        if name in already_yfin:
            continue   # handled by cell-yfinance

        ticker = cik_to_ticker.get(cik, '')
        if not ticker:
            still_no_data.append(f'{name} (no ticker)')
            continue

        print(f'  {name} ({ticker}) ...', end=' ')
        rows = fetch_yfinance_financials(name, ticker)
        if rows:
            fallback_rows.extend(rows)
            print(f'{len(rows)} periods  [Yahoo Finance]')
        else:
            still_no_data.append(name)
            print('no data')

print(f'\nFallback rows added: {len(fallback_rows)}')
if still_no_data:
    print(f'Still no data after fallback: {still_no_data}')

In [ ]:
# Combine SEC EDGAR rows, known-foreign Yahoo Finance rows, and fallback rows
df = pd.DataFrame(all_rows + foreign_rows + fallback_rows)
df['Filing date'] = pd.to_datetime(df['Filing date'])
df = (
    df
    .dropna(subset=['R&D Expenses', 'Revenue', 'Cost of Revenue'], how='all')
    .sort_values(['Company', 'Filing date'])
    .reset_index(drop=True)
)

# Add source and origin columns
df['Data Source'] = df['Data Source'].fillna('SEC EDGAR')
df['Foreign']     = df['Company'].isin(FOREIGN_NAMES)

# Drop companies that never reported any revenue (pre-revenue startups, bad matches)
has_rev = df.groupby('Company')['Revenue'].transform(lambda x: x.notna().any())
no_rev_cos = df.loc[~has_rev, 'Company'].unique().tolist()
if no_rev_cos:
    print(f'Dropping {len(no_rev_cos)} companies with no revenue: {no_rev_cos}')
df = df[has_rev].reset_index(drop=True)

print(f'{len(df):,} rows | {df["Company"].nunique()} companies')
print(f'Foreign companies: {df[df["Foreign"]]["Company"].nunique()}')
print(f'Date range: {df["Filing date"].min().date()} to {df["Filing date"].max().date()}')
df[['Company', 'Foreign', 'Data Source', 'Form Type', 'Filing date', 'Revenue', 'R&D Expenses']].head(30)

In [ ]:
summary = (
    df.groupby('Company')
    .agg(
        filings      = ('Filing date',      'count'),
        first_filing = ('Filing date',      'min'),
        last_filing  = ('Filing date',      'max'),
        has_rnd      = ('R&D Expenses',     lambda x: x.notna().sum()),
        has_revenue  = ('Revenue',          lambda x: x.notna().sum()),
        has_cor      = ('Cost of Revenue',  lambda x: x.notna().sum()),
    )
    .reset_index()
    .sort_values('filings', ascending=False)
)
print(summary.to_string(index=False))

In [ ]:
out = BASE / 'sec_financials.csv'
df[['Company', 'Foreign', 'Data Source', 'Form Type', 'Filing date', 'Period end',
    'R&D Expenses', 'Revenue', 'Cost of Revenue', 'URL']].to_csv(out, index=False)
print(f'Saved {len(df):,} rows to {out}')